In [0]:
# XGBoost — the machine learning model we'll use for churn prediction
# SHAP — explains why the model made each prediction (tells us which factors matter most for churn)
%pip install xgboost shap

In [0]:
# Notebook 4: Churn Prediction
# Food Delivery Analysis

# Importing all the libraries
# --------------------------------------------

from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml import Pipeline
import mlflow
import mlflow.xgboost
import mlflow.sklearn
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, accuracy_score
)
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load user features from Parquet
user_features = spark.read.parquet(
    "/Volumes/workspace/default/food_delivery_data/user_features.parquet"
)

print(f"Loaded {user_features.count():,} users with {len(user_features.columns)} features.")
user_features.display()

In [0]:
# Convert the PySpark dataframe to Pandas
# XGBoost and scikit-learn work with Pandas, not PySpark
user_pd = user_features.toPandas()

# Check the shape
print(f"Dataset shape: {user_pd.shape}")
print(f"\nChurn rate: {user_pd['churn_risk'].mean():.1%}")
print(f"\nColumn types:\n{user_pd.dtypes}")

In [0]:
# Drop user_id as it is just an identifier, not a feature
user_pd = user_pd.drop(columns=["user_id"])

# Convert text columns to numbers using label encoding
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col_name in ["city", "payment_method", "top_cuisine"]:
    user_pd[col_name] = le.fit_transform(user_pd[col_name])

# Separate features and target variable
X = user_pd.drop(columns=["churn_risk"])
y = user_pd["churn_risk"]

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature columns:\n{list(X.columns)}")